# pyAethra example: RMDC26 beginner tier

This notebook runs pyAethra on the **2026 Roman Microlensing Data Challenge (RMDC26) beginner tier**, then writes RogueDB alerts for the objects it flags.

The data are read straight from the Hugging Face dataset [`RGES-PIT/Beginner`](https://huggingface.co/datasets/RGES-PIT/Beginner) with DuckDB, so nothing is downloaded to disk and only the events you select are fetched.

The beginner tier holds 188 events in one long-format parquet file: one row per observation, grouped by `name`, with `bjd`, `filt` (F087 / F146 / F213), `mag`, `mag_err` and the coordinates `ra_deg` / `dec_deg`. `config.yaml` in this folder is set up for those columns.

Requirements: `pip install pyAethra duckdb huggingface_hub matplotlib`

In [ ]:
from pathlib import Path

import numpy as np
import duckdb
import pandas as pd
from huggingface_hub import HfFileSystem

import pyAethra
from pyAethra import load_config, write_alerts, fspl_magnification
from pyAethra.pipeline import run_and_report

print("pyAethra", pyAethra.__version__)

## 1. Connect to the data on Hugging Face

DuckDB reads the parquet file over `hf://` through `HfFileSystem`. If the dataset needs access approval, log in first with `hf auth login`.

In [ ]:
con = duckdb.connect()
con.register_filesystem(HfFileSystem())

URL = "hf://datasets/RGES-PIT/Beginner/RMDC26_Beginner_Tier_test.parquet"
con.sql(f"SELECT count(DISTINCT name) AS events, count(*) AS rows FROM '{URL}'")

## 2. Choose how many events to run

Set `N_EVENTS = None` to run all 188, or a number for a quicker look. Only the rows for the selected events are fetched.

In [ ]:
N_EVENTS = 20

limit = "" if N_EVENTS is None else f"LIMIT {N_EVENTS}"
df = con.sql(f"""
    SELECT * FROM '{URL}'
    WHERE name IN (SELECT DISTINCT name FROM '{URL}' ORDER BY name {limit})
""").df()
print(f"{df['name'].nunique()} events, {len(df):,} rows")

## 3. Configuration

All settings live in `config.yaml`; edit that file to change column names, filters, tuning or the season to run. The beginner tier is in magnitudes, so the pipeline reads `mag_col` / `err_col` and converts to flux internally.

In [ ]:
config = load_config("config.yaml")

## 4. Run the pipeline

`run_and_report` accepts a DataFrame (or a file path), runs the pipeline and prints how many objects each veto fired on and the label counts.

In [ ]:
results = run_and_report(df, config)

In [ ]:
results

## 5. Write alerts

One JSON file per non-flat object, named after the object. Fit values come from the FSPL fit; `rho` is `null` unless finite-source effects are detected.

In [ ]:
paths = write_alerts(results, "alerts")
print(len(paths), "alerts written to", Path("alerts").resolve())
print(paths[0].read_text())

## 6. Plot one candidate with its FSPL model

Takes the candidate with the best FSPL fit and overplots the model on its primary-band data. The model is `Fs * A(t) + Fb`, with `Fs` and `Fb` recovered from `source_mag` and `blend_mag`, converted back to magnitudes for plotting.

In [ ]:
import matplotlib.pyplot as plt

zp = config.get("zero_point", 0.0)
primary = config.get("primary_filter", "F146")

cands = results[results["label"].isin(["microlensing", "ffp"])]
cand = cands.sort_values("chi2_red_fspl").iloc[0]

lc = df[(df["name"] == cand["name"]) & (df["filt"] == primary)]

t0 = cand["t0_fit_fspl"] + 2450000              # output t0 is BJD - 2450000
tE, u0, rho = cand["tE_fit_fspl"], cand["u0_fit_fspl"], cand["rho_fit_fspl"]
fs = 10 ** (-0.4 * (cand["source_mag"] - zp))
fb = 10 ** (-0.4 * (cand["blend_mag"] - zp)) if np.isfinite(cand["blend_mag"]) else 0.0

lo, hi = t0 - 3 * tE, t0 + 3 * tE
win = lc[(lc["bjd"] > lo) & (lc["bjd"] < hi)]
t = np.linspace(lo, hi, 2000)
model_mag = zp - 2.5 * np.log10(fs * fspl_magnification(t, t0, u0, tE, rho) + fb)

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(win["bjd"] - 2450000, win["mag"], win["mag_err"],
            fmt=".", ms=2, alpha=0.4, label=primary)
ax.plot(t - 2450000, model_mag, "k-", lw=1.5, label="FSPL fit")
ax.invert_yaxis()
ax.set_xlabel("BJD - 2450000")
ax.set_ylabel(f"{primary} magnitude")
ax.set_title(f"{cand['name']}  ({cand['label']}, tE = {tE:.1f} d, u0 = {u0:.3f})")
ax.legend()
plt.show()